# 22.11 A/B 测试基础设施 / A/B Testing Infrastructure

**中文**:Part 19 讲过 A/B 测试的**统计学**(怎么设计实验、算样本量、判断显著)。这一节讲**工程基础设施**:当你有一个新模型(比如 22.7 质量门放行的候选),你**怎么真正地把线上流量安全地分给新旧模型**?怎么保证同一个用户不会一会儿看到 A、刷新一下又看到 B?怎么先给 1% 流量试水、没问题再放到 50%?怎么在新模型出事时**秒级回滚**?这些是"把模型安全送上生产"的**流量工程(traffic engineering)**——面试里问"你怎么上线一个新模型"时,真正考的就是这套。本节从零实现 A/B 基础设施的核心——**确定性哈希分桶(deterministic bucketing)**,并讲清影子、金丝雀、蓝绿这几种发布策略。
**English**: Part 19 covered A/B testing's **statistics** (how to design experiments, compute sample size, judge significance). This section covers the **engineering infrastructure**: when you have a new model (say a candidate passed by 22.7's quality gate), **how do you actually and safely split live traffic between the old and new models**? How do you ensure the same user doesn't see A then B on refresh? How do you first try 1% of traffic, then ramp to 50% once it's fine? How do you **roll back in seconds** if the new model misbehaves? This is the **traffic engineering** of "safely shipping a model to production" — when interviews ask "how do you deploy a new model," this is what's really being tested. This section implements A/B infrastructure's core from scratch — **deterministic hash bucketing** — and clarifies shadow, canary, and blue-green release strategies.

---

**中文**:**确定性哈希分桶(deterministic bucketing)—— A/B 基础设施的心脏**。核心需求:给每个用户分配一个变体(control/treatment),要满足:
**English**: **Deterministic hash bucketing — the heart of A/B infrastructure.** Core requirement: assign each user a variant (control/treatment), satisfying:
- **中文**:**一致性(consistency)**:同一个用户**每次**都被分到**同一个变体**(否则用户刷新页面时体验闪来闪去,实验数据也污染了)。
  **Consistency**: the same user is assigned the **same variant every time** (else the experience flickers on refresh and experiment data is polluted).
- **中文**:**无状态(stateless)**:不需要在数据库里存"每个用户分到哪组"——直接从用户 ID **算**出来。做法:`hash(实验ID + 用户ID) % 100` 得到一个 0–99 的桶,按桶落在哪个区间决定变体。
  **Stateless**: no need to store "which group each user is in" in a database — **compute** it from the user ID. Method: `hash(experiment_id + user_id) % 100` gives a bucket 0–99, and which range it falls in decides the variant.
- **中文**:**均匀(uniform)**:好的哈希让流量均匀分到各变体(50/50 就真是各一半)。
  **Uniform**: a good hash splits traffic evenly across variants (50/50 really is half each).
- **中文**:**跨实验独立(independence)**:把**实验ID**也放进哈希,保证一个用户在实验 A 的分组和在实验 B 的分组**互不相关**(否则多个实验会互相干扰)。
  **Independence across experiments**: including the **experiment ID** in the hash ensures a user's group in experiment A is **uncorrelated** with their group in experiment B (else multiple experiments interfere).

> 💡 **面试速查 / Interview cheat-sheet（★★ 上线/实验平台必考）**
> **中文**:**A/B 基础设施**=安全地把流量分给新旧模型/变体。**核心=确定性哈希分桶**:`hash(实验ID+用户ID)%100`→桶→变体; 保证①一致性(同用户永远同组)②无状态(算出来不用存)③均匀④跨实验独立(实验ID入哈希)。**灰度放量(ramp)**:treatment 1%→5%→50%→100%, 每步观察指标和护栏(guardrail)再放大。**发布策略**:①**影子(shadow)**:新模型接收真实流量但**结果不返回给用户**, 只对比记录→零风险验证性能/延迟;②**金丝雀(canary)**:给**小比例真实流量**, 盯紧指标, 有问题快速回滚;③**蓝绿(blue-green)**:新旧两套并存, 流量**瞬间切换**、可秒级回滚;④**滚动(rolling)**:逐个替换实例(接 22.6)。**护栏指标(guardrail)**:除了主指标, 监控延迟/错误率/核心业务不被新模型损害。**别忘**:分流单位(用户 vs 会话 vs 请求)、SUTVA/网络效应(接 19.11)、多重实验隔离。**vs Part 19**:19 讲统计设计与分析, 这里讲流量工程与发布安全。面试金句:*"上线新模型用确定性哈希分桶(hash(实验+用户)%100)保证同用户一致分组、无状态、均匀、跨实验独立; 先影子验证(不影响用户)再金丝雀小流量灰度放量(1%→50%), 配护栏指标监控和秒级回滚(蓝绿/金丝雀); 这是把 22.7 质量门放行的模型安全送上生产的流量工程。"*
> **English**: **A/B infrastructure** = safely split traffic between old/new models/variants. **Core = deterministic hash bucketing**: `hash(experiment_id + user_id) % 100` → bucket → variant; ensuring ① consistency (same user always same group) ② stateless (computed, not stored) ③ uniform ④ independence across experiments (experiment ID in the hash). **Traffic ramp**: treatment 1%→5%→50%→100%, observing metrics and guardrails at each step before scaling up. **Release strategies**: ① **shadow**: the new model receives real traffic but **results aren't returned to users**, only logged for comparison → zero-risk performance/latency validation; ② **canary**: a **small fraction of real traffic**, watch metrics closely, fast rollback on problems; ③ **blue-green**: old and new both exist, traffic **switches instantly**, seconds to roll back; ④ **rolling**: replace instances one by one (ties to 22.6). **Guardrail metrics**: besides the primary metric, monitor latency/error rate/core business not harmed by the new model. **Don't forget**: assignment unit (user vs session vs request), SUTVA/network effects (ties to 19.11), multi-experiment isolation. **vs Part 19**: Part 19 covers statistical design and analysis, here covers traffic engineering and release safety. Interview line: *"Deploy a new model with deterministic hash bucketing (hash(experiment+user)%100) for consistent per-user grouping, stateless, uniform, cross-experiment independent; first validate with shadow (no user impact), then canary small-traffic ramp (1%→50%), with guardrail metrics and seconds-level rollback (blue-green/canary); this is the traffic engineering to safely ship a 22.7-quality-gate-passed model to production."*


In [ ]:

# ============================================================
# 从零实现确定性哈希分桶 / deterministic hash bucketing from scratch
# 中文:A/B 基础设施的心脏。用 hash(实验ID+用户ID)%100 得到一个桶, 决定用户看哪个变体。
#      无需数据库存分组——同一用户每次算出同一个桶(一致), 流量均匀, 跨实验独立。
# English: the heart of A/B infra. hash(experiment_id + user_id) % 100 gives a bucket that decides the user's variant.
#      No database of assignments — the same user computes the same bucket every time (consistent), uniform, cross-experiment independent.
# ============================================================
import hashlib, numpy as np
from collections import Counter
def bucket(user_id, experiment_id, n=100):
    h=hashlib.md5(f"{experiment_id}:{user_id}".encode()).hexdigest()   # 确定性哈希 / deterministic hash
    return int(h,16) % n                                               # 0..99 的桶 / a bucket 0..99
def assign(user_id, experiment_id, split):                            # split: {"control":50,"treatment":50}
    b=bucket(user_id, experiment_id); cum=0
    for variant, pct in split.items():
        cum+=pct
        if b < cum: return variant                                    # 桶落在哪个区间→哪个变体 / bucket range → variant
    return list(split)[-1]

users=[f"user_{i}" for i in range(100000)]
split={"control":50,"treatment":50}
# ① 一致性:同一用户每次同组 / consistency: same user, same variant every time
print("① 一致性 consistency (同用户5次分组一致):",
      all(assign("user_42","exp1",split)==assign("user_42","exp1",split) for _ in range(5)))
# ② 均匀分流 / uniform split
c=Counter(assign(u,"exp1",split) for u in users)
print("② 均匀分流 uniform:", {k:round(v/len(users),3) for k,v in c.items()})
# ③ 跨实验独立:用户在 exp1/exp2 的分组不相关 / independence across experiments
e1=np.array([assign(u,"exp1",split)=="treatment" for u in users])
e2=np.array([assign(u,"exp2",split)=="treatment" for u in users])
print(f"③ 跨实验独立 independence: corr(exp1,exp2)={np.corrcoef(e1,e2)[0,1]:+.3f} (≈0, 互不干扰)")
# ④ 灰度放量:treatment 从 1% 逐步放大 / traffic ramp
print("④ 灰度放量 traffic ramp:")
for pct in [1,5,20,50]:
    frac=np.mean([assign(u,"ramp",{"control":100-pct,"treatment":pct})=="treatment" for u in users])
    print(f"   目标 treatment={pct:>2}% → 实际分流 {frac:.3f}")


In [ ]:

# ============================================================
# 发布策略:影子 / 金丝雀 / 蓝绿(模拟)/ release strategies: shadow / canary / blue-green (simulated)
# 中文:模拟一次新模型上线。影子=新模型接真实流量但结果不返回(只记录对比); 金丝雀=小流量真实服务;
#      蓝绿=瞬间切换+秒级回滚。用一个"新模型偶尔出错"的场景, 看金丝雀如何限制损失范围。
# English: simulate a new-model rollout. Shadow = new model gets real traffic but results aren't served (logged only);
#      canary = small real traffic; blue-green = instant switch + seconds rollback. A "new model occasionally errors" scenario shows how canary limits blast radius.
# ============================================================
np.random.seed(0)
def old_model(x): return 1 if x>0 else 0                    # 稳定的旧模型 / stable old model
def new_model(x):                                          # 新模型:更准, 但有 3% 概率出错(bug)/ new model: better but 3% buggy
    if np.random.rand()<0.03: return None                  # 出故障 / a failure
    return 1 if x>-0.1 else 0
N=20000; xs=np.random.randn(N); uids=[f"u{i}" for i in range(N)]
# 影子:新模型对所有流量预测但不返回给用户, 只统计其故障率 / shadow: predict on all, don't serve, just measure
shadow_failures=sum(new_model(x) is None for x in xs)
print(f"影子模式 shadow:新模型处理全部 {N} 请求但不返回用户 → 观测到故障率 {shadow_failures/N:.1%} (零用户影响)")
# 金丝雀:只有 5% 真实流量走新模型, 故障只影响这 5% / canary: only 5% real traffic → failures limited to 5%
canary_pct=5
served_by_new=[assign(u,"canary",{"old":100-canary_pct,"new":canary_pct})=="new" for u in uids]
canary_failures=sum(new_model(x) is None for x,new in zip(xs,served_by_new) if new)
n_new=sum(served_by_new)
print(f"金丝雀 canary(5% 流量走新模型):{n_new} 个请求, 其中 {canary_failures} 个故障 → 故障波及面仅占全站 {canary_failures/N:.2%}")
print(f"   对比:若直接 100% 切新模型, 故障会波及全站 ~{0.03:.0%} 的请求(金丝雀把爆炸半径限制在 5%)")
print("→ 影子零风险验证 → 金丝雀小流量限损 → 逐步放量 → 蓝绿可秒级回滚, 层层设防安全上线")


In [ ]:

# ============================================================
# 可视化:分流均匀性 + 发布策略的风险 / bucketing uniformity + release-strategy risk
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 100 个桶的分布应近似均匀 / the 100 buckets should be ~uniform
buckets=[bucket(u,"exp1") for u in users]
ax[0].hist(buckets,bins=100,color="#4C72B0",alpha=0.7)
ax[0].axhline(len(users)/100,ls="--",color="#C44E52",label="理论均值(完全均匀)")
ax[0].set_xlabel("桶编号 0-99"); ax[0].set_ylabel("用户数"); ax[0].set_title("确定性哈希→流量均匀落在100个桶"); ax[0].legend(fontsize=8)
# ② 发布策略的爆炸半径 / blast radius of release strategies
strategies=["直接全量\n(100%切换)","金丝雀\n(5%流量)","影子\n(0%用户)"]
blast=[3.0, 3.0*0.05, 0.0]   # % of site affected if new model is 3% buggy
b=ax[1].bar(strategies,blast,color=["#C44E52","#DD8452","#55A868"])
for bar,v in zip(b,blast): ax[1].text(bar.get_x()+bar.get_width()/2,v+0.03,f"{v:.2f}%",ha="center",fontsize=10,weight="bold")
ax[1].set_ylabel("故障波及全站比例 %"); ax[1].set_title("发布策略限制'爆炸半径'(新模型3%故障率)")
plt.tight_layout(); plt.savefig("/tmp/mlops11_viz.png",dpi=80); plt.show()
print("左:好哈希让流量均匀落桶(每桶约1000人); 右:影子/金丝雀把新模型故障的影响范围逐级限制")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **A/B 基础设施的核心,是用"确定性哈希"把随机分组变成无状态、可复现的计算**:一个看似简单的需求——"把用户随机分成两组"——在工程上有微妙的坑:如果你真的用随机数,同一个用户刷新页面就可能换组(体验闪烁、数据污染);如果你在数据库里存"每个用户属于哪组",几亿用户的存储和查询是巨大负担。**确定性哈希分桶**一举解决:`hash(实验ID + 用户ID) % 100`——同一个用户永远算出同一个桶(一致),完全不用存储(无状态),好的哈希保证均匀,把实验ID放进哈希保证多个实验互不干扰(独立)。我们的实验验证了这四条性质全部成立。这是所有实验平台(Google、Meta、字节的实验系统)的共同地基。
2. **安全上线的本质是"限制爆炸半径 + 快速回滚",而非"一次性全切"**:新人常以为"新模型测试通过了就 100% 上线"。但即使离线质量门(22.7)放行,线上仍可能出意外——延迟变高、某类输入触发 bug、和真实流量的交互没测到。成熟的做法是**层层设防的渐进发布**:①**影子模式**——新模型接收真实流量、但结果不返回给用户,你能在**零用户风险**下观测它的真实性能、延迟、故障率;②**金丝雀**——只给一小撮(1%–5%)真实流量,万一新模型有 3% 的故障率,**爆炸半径也被限制在这 5% 之内**(我们的实验清楚地量化了这一点:全量切换会波及全站 3%,金丝雀只波及 0.15%);③**逐步放量**——5%→20%→50%,每步盯着指标和护栏;④**蓝绿**——新旧并存,出事**秒级切回**。这套"渐进 + 可回滚"的哲学,和 22.6 K8s 的滚动更新、22.7 的质量门一脉相承:**永远假设新东西可能出错,并让出错的代价可控。**
3. **诚实的复杂性:基础设施只是"能安全分流",真正的难在实验的科学性**。①**分流单位的选择**是门学问:按**用户**分(同一用户体验一致,适合大多数)、按**会话**分、还是按**请求**分(会导致同一用户看到不同变体,通常不好)?选错会污染实验或恶化体验。②**护栏指标(guardrail)** 至关重要:你在优化主指标(如点击率)时,必须同时监控**不能受损的指标**(延迟、错误率、营收、用户留存)——一个提升点击率但拖慢页面、赶走用户的新模型是净负面。③**这只是"分流",不是"分析"**:正确地把流量分给两组只是第一步,怎么判断"treatment 真的更好"要靠 Part 19 的统计学(样本量、显著性、CUPED、多重检验校正、SUTVA/网络效应)。基础设施保证**分流是干净的**,统计保证**结论是可信的**,两者缺一不可。④**多实验并发**要隔离(层级实验/正交分桶),否则相互干扰。**结论:A/B 基础设施用确定性哈希分桶实现一致、无状态、均匀、独立的流量分配,再用影子→金丝雀→逐步放量→蓝绿的渐进发布限制风险、支持秒级回滚——这是把模型安全送上生产的流量工程;它和 Part 19 的实验统计学互补(一个管分流干净、一个管结论可信),共同构成"科学且安全地上线一个改动"的完整能力。**

**English**:
1. **A/B infrastructure's core is using "deterministic hashing" to turn random grouping into stateless, reproducible computation**: a seemingly simple need — "randomly split users into two groups" — has subtle engineering traps: if you truly use random numbers, the same user may switch groups on refresh (flickering experience, polluted data); if you store "which group each user is in" in a database, storage and lookup for hundreds of millions of users is a huge burden. **Deterministic hash bucketing** solves it at once: `hash(experiment_id + user_id) % 100` — the same user always computes the same bucket (consistent), no storage needed (stateless), a good hash ensures uniformity, and including the experiment ID ensures multiple experiments don't interfere (independent). Our experiment verified all four properties hold. This is the common foundation of all experiment platforms (Google, Meta, ByteDance experiment systems).
2. **Safe deployment's essence is "limiting the blast radius + fast rollback," not "switching all at once"**: newcomers often think "the new model passed testing, so ship it 100%." But even after the offline quality gate (22.7), production can still surprise you — higher latency, a bug triggered by some input class, untested interactions with real traffic. The mature approach is **layered progressive rollout**: ① **shadow mode** — the new model receives real traffic but results aren't served, so you observe its real performance, latency, and failure rate at **zero user risk**; ② **canary** — only a small slice (1%–5%) of real traffic, so if the new model has a 3% failure rate, the **blast radius is limited to that 5%** (our experiment quantified this clearly: a full switch affects 3% of the whole site, canary only 0.15%); ③ **progressive ramp** — 5%→20%→50%, watching metrics and guardrails at each step; ④ **blue-green** — old and new coexist, **switch back in seconds** on trouble. This "progressive + rollback-able" philosophy aligns with 22.6's K8s rolling updates and 22.7's quality gate: **always assume the new thing can fail, and keep the cost of failure controllable.**
3. **Honest complexity: infrastructure only "splits traffic safely"; the real difficulty is the experiment's scientific validity**. ① **Choosing the assignment unit** is an art: split by **user** (consistent experience per user, suits most), by **session**, or by **request** (causing the same user to see different variants, usually bad)? A wrong choice pollutes the experiment or degrades experience. ② **Guardrail metrics** are crucial: while optimizing the primary metric (e.g. click-through), you must simultaneously monitor **metrics that must not be harmed** (latency, error rate, revenue, retention) — a new model that lifts clicks but slows the page and drives users away is net negative. ③ **This is only "splitting," not "analysis"**: correctly splitting traffic is just step one; judging "is treatment truly better" needs Part 19's statistics (sample size, significance, CUPED, multiple-testing correction, SUTVA/network effects). Infrastructure ensures **the split is clean**, statistics ensures **the conclusion is trustworthy**, and both are indispensable. ④ **Concurrent experiments** need isolation (layered experiments/orthogonal bucketing), else they interfere. **Conclusion: A/B infrastructure uses deterministic hash bucketing for consistent, stateless, uniform, independent traffic assignment, then limits risk with shadow→canary→progressive-ramp→blue-green progressive rollout supporting seconds-level rollback — the traffic engineering to safely ship a model to production; it complements Part 19's experiment statistics (one ensures a clean split, the other a trustworthy conclusion), together forming the complete ability to "scientifically and safely ship a change."**

> 💼 **实战视角 / Practical angle**
> **中文**:A/B 基础设施落地:①**确定性哈希分桶** `hash(实验ID+用户ID)%100`——一致、无状态、均匀、跨实验独立(用成熟哈希如 MD5/xxHash);②**渐进发布**:影子(零风险验证性能/延迟)→金丝雀 1-5%(限爆炸半径)→逐步放量→蓝绿(秒级回滚);③**护栏指标**:主指标之外守住延迟/错误率/营收/留存;④**分流单位**通常按用户(体验一致);⑤**和 CI/CD 打通**:22.7 质量门放行→影子→金丝雀→全量, 监控(22.10)异常就自动回滚;⑥并发实验用分层/正交分桶隔离。**平台**:开源如 GrowthBook、Unleash、Flagsmith(特征开关), 大厂自建实验平台; 模型服务层用 KServe/Seldon 的金丝雀能力(接 22.6)。**统计分析**回到 Part 19。面试金句:*"上线新模型用确定性哈希分桶保证同用户一致分组、无状态、均匀、跨实验独立; 用影子(零风险)→金丝雀(小流量限损)→逐步放量→蓝绿(秒级回滚)渐进发布, 配护栏指标和监控自动回滚; 这套流量工程保证'分流干净'、Part 19 的统计保证'结论可信', 二者合起来才是科学又安全的上线。"*
> **English**: A/B infrastructure in practice: ① **deterministic hash bucketing** `hash(experiment_id + user_id) % 100` — consistent, stateless, uniform, cross-experiment independent (use a mature hash like MD5/xxHash); ② **progressive rollout**: shadow (zero-risk performance/latency validation) → canary 1-5% (limit blast radius) → progressive ramp → blue-green (seconds rollback); ③ **guardrail metrics**: beyond the primary metric, protect latency/error rate/revenue/retention; ④ **assignment unit** usually per user (consistent experience); ⑤ **integrate with CI/CD**: 22.7 quality gate passes → shadow → canary → full, auto-rollback on monitoring (22.10) anomalies; ⑥ isolate concurrent experiments with layered/orthogonal bucketing. **Platforms**: open-source like GrowthBook, Unleash, Flagsmith (feature flags), big-tech in-house experiment platforms; at the serving layer use KServe/Seldon's canary capability (ties to 22.6). **Statistical analysis** returns to Part 19. Interview line: *"Deploy a new model with deterministic hash bucketing for consistent per-user grouping, stateless, uniform, cross-experiment independent; roll out progressively with shadow (zero risk) → canary (small-traffic blast-radius limiting) → ramp → blue-green (seconds rollback), with guardrail metrics and auto-rollback on monitoring; this traffic engineering ensures a clean split and Part 19's statistics ensures a trustworthy conclusion — together that's scientific and safe deployment."*

---
### 小结 / Summary
- **中文**:A/B 基础设施=安全分流的流量工程; 核心是确定性哈希分桶(一致、无状态、均匀、跨实验独立)。
- **English**: A/B infrastructure = the traffic engineering for safe splitting; core is deterministic hash bucketing (consistent, stateless, uniform, cross-experiment independent).
- **中文**:渐进发布限风险:影子(零风险)→金丝雀(限爆炸半径)→逐步放量→蓝绿(秒级回滚), 配护栏指标。
- **English**: Progressive rollout limits risk: shadow (zero-risk) → canary (limit blast radius) → progressive ramp → blue-green (seconds rollback), with guardrail metrics.
- **中文**:基础设施管"分流干净", Part 19 统计管"结论可信", 二者互补才是科学且安全的上线。
- **English**: Infrastructure ensures "a clean split," Part 19 statistics ensures "a trustworthy conclusion"; both complement for scientific and safe deployment.
